# TP2 Búsqueda en Pac-Man

Este notebook ejecuta **DFS, BFS, UCS y A*** de `pacman/search.py` con los agentes y el motor originales. Cada búsqueda devuelve acciones para llegar a la meta del laberinto; las partidas se ejecutan sin ventana.

**Uso:** abrir desde Jupyter, reiniciar el kernel y ejecutar todas las celdas en orden. Se necesita la carpeta completa del proyecto, no solo este archivo. En Colab, subir y extraer el ZIP de código y situarse en su carpeta raíz (por ejemplo, con `%cd TP2_Codigo`) antes de ejecutar. No se requieren paquetes adicionales a los del entorno Jupyter/Colab, credenciales ni descargas de datos.

In [1]:
from contextlib import redirect_stdout
from html import escape
from io import StringIO
from pathlib import Path
import os
import sys

from IPython.display import HTML, display

# Admite iniciar el kernel en la raíz del proyecto o en notebooks/.
raiz = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "pacman" / "pacman.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Situá el directorio de trabajo en la raíz del proyecto.")
sys.path.insert(0, str(raiz / "pacman"))
import pacman
import search

In [2]:
def ejecutar(laberinto, funcion, heuristica=None):
    """Juega una partida real y obtiene el camino y sus métricas."""
    directorio_anterior = Path.cwd()
    try:
        os.chdir(raiz / "pacman")
        argumentos = ["-l", laberinto, "-p", "SearchAgent", "-q", "-f"]
        opcion = "fn=" + funcion
        if heuristica:
            opcion += ",heuristic=" + heuristica
        argumentos += ["-a", opcion]
        with redirect_stdout(StringIO()):
            opciones = pacman.readCommand(argumentos)
            jugador = opciones["pacman"]
            crear_problema = jugador.searchType
            problemas = []

            def registrar_problema(estado):
                problema = crear_problema(estado)
                problemas.append(problema)
                return problema

            jugador.searchType = registrar_problema
            partidas = pacman.runGames(**opciones)
        partida = partidas[0]
        assert partida.state.isWin() and not partida.agentCrashed
        assert jugador.actions == [accion for indice, accion in partida.moveHistory
                                   if indice == 0]
        problema = problemas[0]
        return {"Laberinto": laberinto, "Algoritmo": funcion,
                "Heurística": heuristica or "No aplica",
                "Pasos": len(jugador.actions),
                "Costo": problema.getCostOfActions(jugador.actions),
                "Expandidos": problema._expanded, "Resultado": "Victoria",
                "Camino": list(jugador.actions)}
    finally:
        os.chdir(directorio_anterior)


def mostrar(filas):
    columnas = ["Laberinto", "Algoritmo", "Heurística", "Pasos",
                "Costo", "Expandidos", "Resultado"]
    cabecera = "".join("<th>" + escape(c) + "</th>" for c in columnas)
    cuerpo = "".join("<tr>" + "".join(
        "<td>" + escape(str(fila[c])) + "</td>" for c in columnas) + "</tr>"
        for fila in filas)
    display(HTML("<table><thead><tr>" + cabecera +
                 "</tr></thead><tbody>" + cuerpo + "</tbody></table>"))

## Costos unitarios

En `mediumMaze` y `bigMaze` cada paso cuesta 1. Se comparan DFS, BFS, UCS y A* con Manhattan y con heurística nula. **Pasos**, **costo** y **nodos expandidos** se obtienen de cada ejecución; el puntaje del videojuego es una medida distinta.

In [3]:
resultados = []
for laberinto in ["mediumMaze", "bigMaze"]:
    filas = [ejecutar(laberinto, nombre) for nombre in ["dfs", "bfs", "ucs"]]
    filas += [ejecutar(laberinto, "astar", "manhattanHeuristic"),
              ejecutar(laberinto, "astar", "nullHeuristic")]
    dfs, bfs, ucs, astar, nula = filas
    assert bfs["Pasos"] == ucs["Pasos"] == astar["Pasos"] == nula["Pasos"]
    assert bfs["Costo"] == ucs["Costo"] == astar["Costo"] == nula["Costo"]
    assert nula["Expandidos"] == ucs["Expandidos"]
    resultados.extend(filas)
mostrar(resultados)

Laberinto,Algoritmo,Heurística,Pasos,Costo,Expandidos,Resultado
mediumMaze,dfs,No aplica,130,130,146,Victoria
mediumMaze,bfs,No aplica,68,68,269,Victoria
mediumMaze,ucs,No aplica,68,68,269,Victoria
mediumMaze,astar,manhattanHeuristic,68,68,221,Victoria
mediumMaze,astar,nullHeuristic,68,68,269,Victoria
bigMaze,dfs,No aplica,210,210,390,Victoria
bigMaze,bfs,No aplica,210,210,620,Victoria
bigMaze,ucs,No aplica,210,210,620,Victoria
bigMaze,astar,manhattanHeuristic,210,210,549,Victoria
bigMaze,astar,nullHeuristic,210,210,620,Victoria


## Caminos obtenidos

En el laberinto pequeño `tinyMaze` se muestran las listas completas de acciones. DFS puede elegir una ruta más larga; BFS minimiza pasos y UCS/A* minimizan costo bajo sus condiciones correspondientes.

In [4]:
caminos = [ejecutar("tinyMaze", nombre,
                    "manhattanHeuristic" if nombre == "astar" else None)
           for nombre in ["dfs", "bfs", "ucs", "astar"]]
for fila in caminos:
    print(f'{fila["Algoritmo"]}: {fila["Pasos"]} pasos, costo {fila["Costo"]}')
    print(fila["Camino"])
    print()

dfs: 10 pasos, costo 10
['West', 'West', 'West', 'West', 'South', 'South', 'East', 'South', 'South', 'West']

bfs: 8 pasos, costo 8
['South', 'South', 'West', 'South', 'West', 'West', 'South', 'West']

ucs: 8 pasos, costo 8
['South', 'South', 'West', 'South', 'West', 'West', 'South', 'West']

astar: 8 pasos, costo 8
['South', 'South', 'West', 'South', 'West', 'West', 'South', 'West']



## Costos variables

Se conserva el problema del agente original `StayWestSearchAgent`: entrar a una posición `(x, y)` cuesta `2 ** x`. BFS y UCS utilizan **la misma función de costo**. Aquí se comparan planes de posición hasta la meta `(1, 1)`, sin evaluar enfrentamientos con fantasmas. Cada camino calculado se verifica contra las paredes del laberinto. Manhattan no se usa en este caso.


In [5]:
import layout
import searchAgents
from game import Actions

directorio_anterior = Path.cwd()
try:
    os.chdir(raiz / "pacman")
    escenario = layout.getLayout("mediumScaryMaze")
finally:
    os.chdir(directorio_anterior)
estado_inicial = pacman.GameState()
estado_inicial.initialize(escenario, escenario.getNumGhosts())
agente_ponderado = searchAgents.StayWestSearchAgent()
ponderados = []
for nombre in ["bfs", "ucs"]:
    problema = agente_ponderado.searchType(estado_inicial)
    camino = getattr(search, nombre)(problema)
    expandidos = problema._expanded
    posicion = problema.getStartState()
    for accion in camino:
        dx, dy = Actions.directionToVector(accion)
        posicion = (int(posicion[0] + dx), int(posicion[1] + dy))
        assert not problema.walls[posicion[0]][posicion[1]]
    assert problema.isGoalState(posicion)
    ponderados.append({"Laberinto": "mediumScaryMaze", "Algoritmo": nombre,
                       "Heurística": "No aplica", "Pasos": len(camino),
                       "Costo": problema.getCostOfActions(camino),
                       "Expandidos": expandidos, "Resultado": "Meta alcanzada",
                       "Camino": camino})
bfs, ucs = ponderados
assert ucs["Costo"] < bfs["Costo"]
assert ucs["Pasos"] > bfs["Pasos"]
mostrar(ponderados)
print(f'UCS utiliza {ucs["Pasos"] - bfs["Pasos"]} pasos más y reduce '
      f'el costo en {bfs["Costo"] - ucs["Costo"]}.')


Laberinto,Algoritmo,Heurística,Pasos,Costo,Expandidos,Resultado
mediumScaryMaze,bfs,No aplica,72,70670371838,279,Meta alcanzada
mediumScaryMaze,ucs,No aplica,92,68719479864,108,Meta alcanzada


UCS utiliza 20 pasos más y reduce el costo en 1950891974.


## Comprobaciones

Las aserciones anteriores verifican victorias, la ejecución del camino completo, la igualdad de costos en los laberintos unitarios y la legalidad de los planes ponderados. Las funciones de búsqueda se importan del proyecto; no se vuelven a implementar en este notebook.

Para ejecutar la batería de casos límite e integración, usar desde la raíz del proyecto:

```text
python -m unittest discover -s tests -v
```

Código educativo de UC Berkeley, adaptado para UADE por Nelson Ponzoni. Se mantienen los avisos originales en `pacman/`: <http://ai.berkeley.edu>.